In [ ]:
# ============================
# IMPORTS
# ============================
import os
import re
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, classification_report
from transformers import AutoTokenizer, AutoModel
import torch
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================
# PREPROCESSING DATA
# ============================
# Atur folder path
DATA_DIR = "/content/drive/My Drive/SEC/Data DSC/data"
TEXT_DIR = "/content/drive/My Drive/SEC/Data DSC/papers"

# Load Dataset
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
meta_df = pd.read_csv(os.path.join(DATA_DIR, "papers_metadata.csv"))

# Merge metadata untuk paper dan referenced_paper
def merge_metadata(pair_df):
    df = pair_df.copy()
    df = df.merge(meta_df.add_prefix("paper_"), how="left", left_on="paper", right_on="paper_paper_id")
    df = df.merge(meta_df.add_prefix("ref_"), how="left", left_on="referenced_paper", right_on="ref_paper_id")
    return df

train_df = merge_metadata(train_df)
test_df = merge_metadata(test_df)

for col in ["paper_title", "ref_title"]:
    train_df[col] = train_df[col].fillna("")
    test_df[col] = test_df[col].fillna("")


# Simpan Output
output_dir = '/content/drive/My Drive/SEC/Data DSC/'

train_df.to_csv(os.path.join(output_dir, "train_preprocessed.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test_preprocessed.csv"), index=False)

print("✅ Data preparation selesai! Output disimpan di:", output_dir)

✅ Data preparation selesai! Output disimpan di: /content/drive/My Drive/SEC/Data DSC/


In [ ]:
# ============================
# LOAD & ENCODE TEXTS DENGAN SPECTER2
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoModel.from_pretrained("allenai/specter2_base").to(device)

@torch.no_grad()
def encode_specter(texts, batch_size=16):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding Specter2"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
        output = model(**inputs)
        cls = output.last_hidden_state[:, 0]  # [CLS]
        embeddings.append(cls.cpu().numpy())
    return np.vstack(embeddings)

# Load teks langsung dari folder .txt
folder_path = "/content/drive/My Drive/SEC/Data DSC/papers"
paper_text_map = {}

for fname in tqdm(os.listdir(folder_path), desc="Loading paper text"):
    if fname.endswith(".txt"):
        paper_id = fname.replace(".txt", "")
        try:
            with open(os.path.join(folder_path, fname), "r", encoding="utf-8") as f:
                paper_text_map[paper_id] = f.read().strip()
        except:
            paper_text_map[paper_id] = ""

paper_ids = list(set(paper_text_map.keys()))
paper_texts = [paper_text_map[pid] for pid in paper_ids]

# Encode
paper_embs = encode_specter(paper_texts)
embed_dict = dict(zip(paper_ids, paper_embs))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/717k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Encoding Specter2: 100%|██████████| 273/273 [02:11<00:00,  2.08it/s]


In [ ]:
# ============================
# LOAD DATA
# ============================
train_df = pd.read_csv("/content/drive/My Drive/SEC/Data DSC/train_preprocessed.csv")
test_df = pd.read_csv("/content/drive/My Drive/SEC/Data DSC/test_preprocessed.csv")
# ============================
# FEATURE CONSTRUCTION (TRAIN & TEST)
# ============================
def extract_metadata_features(df):
    feat = []
    for row in df.itertuples():
        yd = row.paper_publication_year - row.ref_publication_year if pd.notna(row.paper_publication_year) and pd.notna(row.ref_publication_year) else 999
        future = int(yd < 0)
        same_type = int(row.paper_type == row.ref_type)
        cited_p = row.paper_cited_by_count if pd.notna(row.paper_cited_by_count) else 0
        cited_r = row.ref_cited_by_count if pd.notna(row.ref_cited_by_count) else 0
        feat.append([yd, future, same_type, cited_p, cited_r])
    return np.array(feat)

def build_features(df):
    X, y = [], []
    for row in tqdm(df.itertuples(), total=len(df), desc="Building features"):
        pid, rid = row.paper, row.referenced_paper
        emb1, emb2 = embed_dict.get(pid), embed_dict.get(rid)
        if emb1 is None or emb2 is None:
            continue
        concat = np.concatenate([emb1, emb2])
        diff = np.abs(emb1 - emb2)
        features = np.concatenate([concat, diff])
        X.append(features)
        if hasattr(row, "is_referenced"):
            y.append(row.is_referenced)
    return np.array(X), np.array(y) if y else None

X_specter_train, y_train = build_features(train_df)
X_specter_test, _ = build_features(test_df)
X_meta_train = extract_metadata_features(train_df)
X_meta_test = extract_metadata_features(test_df)

X_train = np.hstack([X_specter_train, X_meta_train])
X_test = np.hstack([X_specter_test, X_meta_test])


Building features: 100%|██████████| 336021/336021 [00:03<00:00, 88684.37it/s]


In [ ]:
# ============================
# SCALING & SPLIT
# ============================
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Simpan scaler
joblib.dump(scaler, "/content/drive/My Drive/SEC/Data DSC/scaler.pkl")

# ============================
# MODEL DEFINITION
# ============================
model = Sequential([
    Dense(768, input_shape=(X_tr_scaled.shape[1],)),
    BatchNormalization(),
    LeakyReLU(alpha=0.01),
    Dropout(0.3),
    Dense(512),
    BatchNormalization(),
    LeakyReLU(alpha=0.01),
    Dropout(0.3),
    Dense(256),
    BatchNormalization(),
    LeakyReLU(alpha=0.01),
    Dropout(0.2),
    Dense(128),
    BatchNormalization(),
    LeakyReLU(alpha=0.01),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# ============================
# TRAINING
# ============================
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]
model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val), epochs=20, batch_size=64, callbacks=callbacks)

# Simpan model
model.save("/content/drive/My Drive/SEC/Data DSC/specter2_nn_model.h5")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9778 - loss: 0.1116 - val_accuracy: 0.9904 - val_loss: 0.0310 - learning_rate: 1.0000e-04
Epoch 2/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9898 - loss: 0.0334 - val_accuracy: 0.9902 - val_loss: 0.0306 - learning_rate: 1.0000e-04
Epoch 3/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9905 - loss: 0.0293 - val_accuracy: 0.9909 - val_loss: 0.0288 - learning_rate: 1.0000e-04
Epoch 4/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9909 - loss: 0.0266 - val_accuracy: 0.9910 - val_loss: 0.0279 - learning_rate: 1.0000e-04
Epoch 5/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9912 - loss: 0.0244 - val_accuracy: 0.9910 - val_loss: 0.0274 - learning_rate: 1.0000e-04
Epoch 6/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9917 - loss: 0.0223 - val_accuracy: 0.9911 - val_loss: 0.0277 - learning_rate: 1.0000e-04
Epoch 7/20
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 17s 

In [ ]:
# ============================
# THRESHOLD TUNING & EVALUATION
# ============================
y_val_proba = model.predict(X_val_scaled)

best_thresh = 0.5
best_mcc = -1
for t in np.arange(0, 1.01, 0.01):
    y_pred = (y_val_proba >= t).astype(int)
    mcc = matthews_corrcoef(y_val, y_pred)
    if mcc > best_mcc:
        best_mcc = mcc
        best_thresh = t

print(f"\n🔍 Best Threshold: {best_thresh:.2f} with MCC: {best_mcc:.4f}")
final_pred = (y_val_proba >= best_thresh).astype(int)
print(classification_report(y_val, final_pred))


2567/2567 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step

🔍 Best Threshold: 0.35 with MCC: 0.5306
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     81281
           1       0.55      0.52      0.54       858

    accuracy                           0.99     82139
   macro avg       0.77      0.76      0.77     82139
weighted avg       0.99      0.99      0.99     82139



In [ ]:
# ============================
# PREDICT TEST
# ============================
y_test_proba = model.predict(X_test_scaled)
y_test_pred = (y_test_proba >= best_thresh).astype(int)

submission = test_df[["id"]].copy()
submission["is_referenced"] = y_test_pred
submission.to_csv("/content/drive/My Drive/SEC/Data DSC/submission6.csv", index=False)
print("\n✅ Submission saved to submission.csv")

10501/10501 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step

✅ Submission saved to submission.csv
